# Manage View Layers & Web Maps

Creates **four hosted view layers** and corresponding **web maps** from an existing field data collection layer.

| # | View | Definition Query | Editable | Sharing |
|---|------|-----------------|----------|---------|
| 1 | **Collector** | *(no filter — all features)* | Create + Update | Public |
| 2 | **Approver** | `review_status IN ('Pending Review', 'In Review', 'Rejected', 'Approved')` | Update | Org |
| 3 | **Public** | `review_status = 'Approved'` | None (read-only) | Public |
| 4 | **Pending** | `review_status <> 'Approved' OR review_status IS NULL` | None (read-only) | Public |

The **Pending** view is added as a grey-dot layer underneath the main layer on both the **Public** and **Approver** web maps — no popups, no visible fields.

All reusable logic lives in `views/manage.py`; this notebook is orchestration only.

## Step 1 – Connect & Load Base Layer

In [24]:
import getpass, json, os, sys, urllib3
from pathlib import Path
from arcgis.gis import GIS
from arcgis.features import FeatureLayerCollection

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# ── Configuration ──────────────────────────────────────────────────────
# Set STAGING to True for test runs — uses a separate registry and adds
# " - STAGING" to all item titles so production items are never touched.
STAGING = True

# Web map whose Field Maps config (Arcade expressions, form layout,
# templates) will be copied into the Collector & Approver maps.
# Set to None to skip the copy step and configure manually.
FIELDMAPS_TEMPLATE_WM_ID = "a344064c381345d49cb4d674976d17eb"
# ───────────────────────────────────────────────────────────────────────

# Ensure repo root is on sys.path so `views` package is importable
_repo_root = os.path.abspath("..")
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

import views  # noqa: E402

username = input("Username: ").strip()
password = getpass.getpass("Password: ")
gis = GIS("https://www.arcgis.com", username, password, verify_cert=False)
print(f"Logged in as {gis.users.me.username} ({gis.url})")

# Choose registry & base layer based on staging mode
if STAGING:
    _registry_path = Path("..") / "item_registry_staging.json"
    _suffix = " - STAGING"
    _reg = views.load_registry(_registry_path)
    item_id = _reg["base_layer"]["item_id"]
    print(f"⚠️  STAGING MODE — registry: {_registry_path.name}")
else:
    _registry_path = Path("..") / "item_registry.json"
    _suffix = ""
    _reg = views.load_registry(_registry_path)
    item_id = _reg["base_layer"]["item_id"]
    print(f"PRODUCTION MODE — registry: {_registry_path.name}")

print(f"Base layer ID: {item_id}")

flc_item = gis.content.get(item_id)
flc = FeatureLayerCollection.fromitem(flc_item)
base_layer = flc.layers[0]
source_folder = flc_item.ownerFolder
_clean = views.clean_title(flc_item.title)

print(f"Loaded: {flc_item.title}  |  Features: {base_layer.query(return_count_only=True)}")
print(f"Fields: {[f['name'] for f in base_layer.properties.fields]}")
print(f"Folder: {source_folder or '(Root)'}")

Setting `verify_cert` to False is a security risk, use at your own risk.


Logged in as ralouta.aiddev (https://EsriAidDev.maps.arcgis.com)
⚠️  STAGING MODE — registry: item_registry_staging.json
Base layer ID: d7219e80879b42c98ddc7cda1a0369ff
Loaded: Plant Identification AI STAGING  |  Features: 3183
Fields: ['OBJECTID', 'GlobalID', 'common_name', 'latin_name', 'family', 'plant_type', 'condition', 'height_m', 'canopy_width_m', 'observation_date', 'observer', 'notes', 'review_status', 'reviewed_by', 'review_date', 'revision_notes', 'created_user', 'created_date', 'last_edited_user', 'last_edited_date']
Folder: 1657ba943f4545198c54797359dbb665


## Step 2 – Configure Field Visibility per View

Select a view from the dropdown, then choose which fields should be **visible** and which should be **read-only**. System fields (`objectid`, `globalid`) are always included. The Pending view hides all fields automatically.

In [25]:
import ipywidgets as widgets
from IPython.display import display

# Build default field config from views module
_view_field_config, _selectable_fields = views.build_field_config(base_layer)

# ── Build interactive widget ──────────────────────────────────────────
_view_dropdown = widgets.Dropdown(
    options=list(_view_field_config.keys()),
    value="Collector",
    description="View:",
    layout=widgets.Layout(width="300px"),
)

_header = widgets.HBox([
    widgets.Label(value="Field", layout=widgets.Layout(width="200px")),
    widgets.Label(value="Visible", layout=widgets.Layout(width="100px")),
    widgets.Label(value="Read-only", layout=widgets.Layout(width="100px")),
])

_field_checkboxes = {}
_field_rows = []
for fname in _selectable_fields:
    vis_cb = widgets.Checkbox(value=False, indent=False, layout=widgets.Layout(width="100px"))
    ro_cb = widgets.Checkbox(value=False, indent=False, layout=widgets.Layout(width="100px"))

    def _make_vis_handler(v_cb, r_cb):
        def _handler(change):
            if not change["new"]:
                r_cb.value = False
            r_cb.disabled = not change["new"]
        return _handler
    vis_cb.observe(_make_vis_handler(vis_cb, ro_cb), names="value")

    def _make_save_handler(field_name, v_cb, r_cb):
        def _handler(change):
            vname = _view_dropdown.value
            _view_field_config[vname][field_name]["visible"] = v_cb.value
            _view_field_config[vname][field_name]["readonly"] = r_cb.value
        return _handler
    vis_cb.observe(_make_save_handler(fname, vis_cb, ro_cb), names="value")
    ro_cb.observe(_make_save_handler(fname, vis_cb, ro_cb), names="value")

    _field_checkboxes[fname] = (vis_cb, ro_cb)
    _field_rows.append(widgets.HBox([
        widgets.Label(value=fname, layout=widgets.Layout(width="200px")), vis_cb, ro_cb,
    ]))

_fields_container = widgets.VBox([_header] + _field_rows)

def _load_view_config(change=None):
    vname = _view_dropdown.value
    for fname, (vis_cb, ro_cb) in _field_checkboxes.items():
        cfg = _view_field_config[vname][fname]
        vis_cb.value = cfg["visible"]
        ro_cb.value = cfg["readonly"]
        ro_cb.disabled = not cfg["visible"]

_view_dropdown.observe(_load_view_config, names="value")
_load_view_config()

print("Configure field visibility per view layer (Pending hides all fields automatically):\n")
display(widgets.VBox([_view_dropdown, _fields_container]))

Configure field visibility per view layer (Pending hides all fields automatically):



## Step 3 – Create All Views (Collector, Approver, Public, Pending)

In [26]:
# Print resolved field config summary
for vname in _view_field_config:
    visible = sorted(k for k, v in _view_field_config[vname].items() if v["visible"])
    readonly = sorted(k for k, v in _view_field_config[vname].items() if v["readonly"])
    print(f"{vname} visible: {visible}")
    if readonly:
        print(f"{vname} read-only: {readonly}")

# ── 1. Collector ──────────────────────────────────────────────────────
collector_view_item, collector_lyr = views.create_view(
    gis, flc, base_layer,
    f"{_clean} - Collector{_suffix}",
    updateable=True,
    capabilities="Create,Query,Update,Delete,Editing,Sync",
    description="Field Maps collection view. Shows all features for field data collection.",
    tags=f"{_clean}, Field Collection, Field Maps, Collector",
    snippet=f"Collector view for {_clean}",
    folder=source_folder,
    field_updates=views.get_field_updates(
        base_layer, _view_field_config["Collector"], editable_view=True),
)
print(f"✓ Collector view: {collector_view_item.title}")

# ── 2. Approver ──────────────────────────────────────────────────────
approver_view_item, approver_lyr = views.create_view(
    gis, flc, base_layer,
    f"{_clean} - Approver{_suffix}",
    updateable=True,
    capabilities="Create,Query,Update,Delete,Editing",
    description="Approver view for reviewing, approving, and rejecting features.",
    tags=f"{_clean}, Approver, Review, AI",
    snippet=f"Approver view for {_clean}",
    folder=source_folder,
    def_query="review_status IN ('Pending Review', 'In Review', 'Rejected', 'Approved')",
    field_updates=views.get_field_updates(
        base_layer, _view_field_config["Approver"], editable_view=True),
)
print(f"✓ Approver view: {approver_view_item.title}")

# ── 3. Public (approved, read-only) ──────────────────────────────────
public_view_item, public_lyr = views.create_view(
    gis, flc, base_layer,
    f"{_clean} - Public{_suffix}",
    updateable=False,
    capabilities="Query,Extract",
    description="Read-only public view of approved records. User/review fields hidden.",
    tags=f"{_clean}, Public, Approved",
    snippet=f"Public approved data view for {_clean}",
    folder=source_folder,
    def_query="review_status = 'Approved'",
    field_updates=views.get_field_updates(
        base_layer, _view_field_config["Public"], editable_view=False),
)
print(f"✓ Public view: {public_view_item.title}")

# ── 4. Pending (non-approved, read-only, grey dots) ──────────────────
pending_view_item, pending_lyr = views.create_view(
    gis, flc, base_layer,
    f"{_clean} - Pending{_suffix}",
    updateable=False,
    capabilities="Query",
    description="Read-only public view of non-approved (pending) records. Grey dots, no popups.",
    tags=f"{_clean}, Public, Pending",
    snippet=f"Pending records for {_clean} (grey dots, no popups)",
    folder=source_folder,
    def_query="review_status <> 'Approved' OR review_status IS NULL",
    field_updates=views.get_hidden_field_updates(base_layer),
    renderer=views.build_pending_renderer(),
)
print(f"✓ Pending view: {pending_view_item.title}")

print("\n── All 4 views created ──")

Collector visible: ['notes', 'observation_date', 'observer', 'review_status', 'revision_notes']
Approver visible: ['canopy_width_m', 'common_name', 'condition', 'created_date', 'created_user', 'family', 'height_m', 'last_edited_date', 'last_edited_user', 'latin_name', 'notes', 'observation_date', 'observer', 'plant_type', 'review_date', 'review_status', 'reviewed_by', 'revision_notes']
Public visible: ['canopy_width_m', 'common_name', 'condition', 'family', 'height_m', 'latin_name', 'notes', 'observation_date', 'plant_type']
  Created view item: 5b3fece5247b4862ae7009f1aabd44b6
✓ Collector view: Plant Identification AI STAGING - Collector - STAGING
  Created view item: ed533487ea0c4877a71d14e591a40322
✓ Approver view: Plant Identification AI STAGING - Approver - STAGING
  Created view item: 367a69b3a3a747e5a56d4d9504b1d024
✓ Public view: Plant Identification AI STAGING - Public - STAGING
  Created view item: 8bcb8b3051ec484fb783b69542a0b33b
✓ Pending view: Plant Identification AI STAGI

## Step 4 – Set Sharing & Apply Renderers

- **Collector** + **Public** + **Pending** → Public
- **Approver** → Org (internal workflow)

In [27]:
# ── Sharing ────────────────────────────────────────────────────────────
approver_view_item.sharing.sharing_level = "ORG"
print("Approver sharing: ORG")

for _label, _item in [
    ("Collector", collector_view_item),
    ("Public", public_view_item),
    ("Pending", pending_view_item),
]:
    _item.sharing.sharing_level = "EVERYONE"
    print(f"{_label} sharing: EVERYONE")

# ── Renderers ─────────────────────────────────────────────────────────
collector_renderer = views.build_status_renderer()
approver_renderer = views.build_status_renderer()
public_renderer = views.build_single_renderer([56, 168, 0, 140])

for _label, _v_item, _rend in [
    ("Collector", collector_view_item, collector_renderer),
    ("Approver", approver_view_item, approver_renderer),
    ("Public", public_view_item, public_renderer),
]:
    _v_item, _lyr = views.get_view_layer(_v_item, gis)
    _lyr.manager.update_definition({"drawingInfo": {"renderer": _rend}})
    if _label == "Collector":
        collector_view_item, collector_lyr = _v_item, _lyr
    elif _label == "Approver":
        approver_view_item, approver_lyr = _v_item, _lyr
    elif _label == "Public":
        public_view_item, public_lyr = _v_item, _lyr
    print(f"✓ {_label} renderer applied")

# ── Attachments ───────────────────────────────────────────────────────
if not getattr(base_layer.properties, "hasAttachments", False):
    base_layer.manager.update_definition({"hasAttachments": True})
    print("→ Enabled attachments on SOURCE layer")

for _label, _v_item in [("Collector", collector_view_item), ("Approver", approver_view_item)]:
    _v_item, _lyr = views.get_view_layer(_v_item, gis)
    if not getattr(_lyr.properties, "hasAttachments", False):
        _lyr.manager.update_definition({"hasAttachments": True})
        print(f"→ Enabled attachments on {_label} view")

# ── Verify Collector editing capabilities ──────────────────────────────
collector_flc = FeatureLayerCollection.fromitem(gis.content.get(collector_view_item.id))
_svc_caps = collector_flc.properties.get("capabilities", "")
_required = {"Create", "Query", "Update", "Delete", "Editing", "Sync"}
_current = {c.strip() for c in _svc_caps.split(",")}
if not _required.issubset(_current):
    _new_caps = ",".join(sorted(_required | _current))
    collector_flc.manager.update_definition({"capabilities": _new_caps})
    print(f"→ Updated Collector service capabilities to: {_new_caps}")
else:
    print(f"→ Collector service capabilities OK: {_svc_caps}")

Approver sharing: ORG
Collector sharing: EVERYONE
Public sharing: EVERYONE
Pending sharing: EVERYONE
✓ Collector renderer applied
✓ Approver renderer applied
✓ Public renderer applied
→ Collector service capabilities OK: Create,Delete,Query,Update,Editing,Sync


## Step 5 – Create Web Maps & Add Pending Layer

In [28]:
# Re-fetch all view layers for URL access
collector_view_item, collector_lyr = views.get_view_layer(collector_view_item, gis)
approver_view_item, approver_lyr = views.get_view_layer(approver_view_item, gis)
public_view_item, public_lyr = views.get_view_layer(public_view_item, gis)

# 1. Collector Web Map — editable, offline-enabled
collector_wm = views.create_webmap(
    gis, collector_view_item, collector_lyr,
    f"{_clean} - Collector Map{_suffix}",
    f"Field Maps collection map for {_clean}.",
    _clean,
    editable=True, offline=True, folder=source_folder,
    renderer=collector_renderer,
    field_config=_view_field_config["Collector"],
)
print(f"✓ Collector Map: {collector_wm.title}  (ID: {collector_wm.id})")

# 2. Approver Web Map — editable
approver_wm = views.create_webmap(
    gis, approver_view_item, approver_lyr,
    f"{_clean} - Approver Map{_suffix}",
    f"Approver map — review, approve, reject records for {_clean}.",
    _clean,
    editable=True, folder=source_folder,
    renderer=approver_renderer,
    field_config=_view_field_config["Approver"],
)
print(f"✓ Approver Map: {approver_wm.title}  (ID: {approver_wm.id})")

# 3. Public Web Map — read-only
public_wm = views.create_webmap(
    gis, public_view_item, public_lyr,
    f"{_clean} Map{_suffix}",
    f"Public read-only map of approved records for {_clean}.",
    _clean,
    editable=False, folder=source_folder,
    renderer=public_renderer,
    field_config=_view_field_config["Public"],
)
print(f"✓ Public Map: {public_wm.title}  (ID: {public_wm.id})")

# 4. Add pending layer (grey dots) to Public + Approver web maps
pending_renderer = views.build_pending_renderer()
for _label, _wm in [("Public", public_wm), ("Approver", approver_wm)]:
    views.add_pending_to_webmap(
        gis, _wm.id, pending_view_item, pending_lyr,
        pending_renderer, _clean,
    )
    print(f"✓ Pending layer added to {_label} web map")

✓ Collector Map: Plant Identification AI STAGING - Collector Map - STAGING  (ID: 28af50c3c72d42acaa8717ae4bb4888e)
✓ Approver Map: Plant Identification AI STAGING - Approver Map - STAGING  (ID: e869847b21ee40568296d4f9180aad1d)
✓ Public Map: Plant Identification AI STAGING Map - STAGING  (ID: 9a804959bd7543beb9abc4a95869b87d)
✓ Pending layer added to Public web map
✓ Pending layer added to Approver web map


In [29]:
# ── Copy Field Maps config from template web map (Arcade, form, templates) ──
if FIELDMAPS_TEMPLATE_WM_ID:
    print(f"Copying Field Maps config from template map ({FIELDMAPS_TEMPLATE_WM_ID})...")

    for _label, _wm in [("Collector", collector_wm), ("Approver", approver_wm)]:
        _updated, _keys = views.copy_fieldmaps_config(gis, FIELDMAPS_TEMPLATE_WM_ID, _wm.id)
        if _keys:
            print(f"  ✓ {_label} Map: copied {', '.join(_keys)}")
        else:
            print(f"  ⚠️ {_label} Map: no Field Maps config found in template")
else:
    print("No FIELDMAPS_TEMPLATE_WM_ID set — configure Field Maps Arcade manually")

Copying Field Maps config from template map (a344064c381345d49cb4d674976d17eb)...
  ✓ Collector Map: copied formInfo
  ✓ Approver Map: copied formInfo


In [30]:
# ── Share web maps ─────────────────────────────────────────────────────
approver_wm.sharing.sharing_level = "ORG"
print("Approver Map sharing: ORG")

for _label, _wm in [("Collector", collector_wm), ("Public", public_wm)]:
    _wm.sharing.sharing_level = "EVERYONE"
    print(f"{_label} Map sharing: EVERYONE")

Approver Map sharing: ORG
Collector Map sharing: EVERYONE
Public Map sharing: EVERYONE


In [31]:
# ── Generate thumbnails ────────────────────────────────────────────────
for _label, _wm in [("Collector", collector_wm), ("Approver", approver_wm), ("Public", public_wm)]:
    _thumb_label = f"{_label}{_suffix}" if _suffix else _label
    print(f"  {_thumb_label} thumbnail ... ", end="", flush=True)
    try:
        ok = views.set_thumbnail(gis, _wm, _thumb_label)
        print("✓" if ok else "failed (see response above)")
    except Exception as e:
        print(f"⚠️ {e}")
print("Done.")

  Collector - STAGING thumbnail ... ✓
  Approver - STAGING thumbnail ... ✓
  Public - STAGING thumbnail ... ✓
Done.


## Step 6 – Summary & Save Registry

In [32]:
print("=" * 70)
print("VIEW LAYERS & WEB MAPS CREATED")
print("=" * 70)

_all_items = [
    ("Collector", collector_view_item, collector_wm, "Public"),
    ("Approver",  approver_view_item,  approver_wm,  "Org"),
    ("Public",    public_view_item,    public_wm,    "Public"),
    ("Pending",   pending_view_item,   None,         "Public"),
]

for label, v_item, wm_item, sharing in _all_items:
    print(f"\n── {label} ──")
    print(f"  View Layer:  {v_item.title}")
    print(f"  View ID:     {v_item.id}")
    if wm_item:
        print(f"  Web Map:     {wm_item.title}")
        print(f"  Web Map ID:  {wm_item.id}")
    else:
        print(f"  Web Map:     (added to Public + Approver web maps)")
    print(f"  Sharing:     {sharing}")

print("\n── Next Steps ──")
print("1. On the Collector view, add a Field Maps Arcade calculation on review_status → 'Pending Review'")
print("2. Configure feature templates manually in Field Maps")
print("3. Run the AI enrichment notebook targeting the Approver view")
print("4. Use the Approver Map to approve / reject / send back records")

VIEW LAYERS & WEB MAPS CREATED

── Collector ──
  View Layer:  Plant Identification AI STAGING - Collector - STAGING
  View ID:     5b3fece5247b4862ae7009f1aabd44b6
  Web Map:     Plant Identification AI STAGING - Collector Map - STAGING
  Web Map ID:  28af50c3c72d42acaa8717ae4bb4888e
  Sharing:     Public

── Approver ──
  View Layer:  Plant Identification AI STAGING - Approver - STAGING
  View ID:     ed533487ea0c4877a71d14e591a40322
  Web Map:     Plant Identification AI STAGING - Approver Map - STAGING
  Web Map ID:  e869847b21ee40568296d4f9180aad1d
  Sharing:     Org

── Public ──
  View Layer:  Plant Identification AI STAGING - Public - STAGING
  View ID:     367a69b3a3a747e5a56d4d9504b1d024
  Web Map:     Plant Identification AI STAGING Map - STAGING
  Web Map ID:  9a804959bd7543beb9abc4a95869b87d
  Sharing:     Public

── Pending ──
  View Layer:  Plant Identification AI STAGING - Pending - STAGING
  View ID:     8bcb8b3051ec484fb783b69542a0b33b
  Web Map:     (added to Public 

In [33]:
registry = views.load_registry(_registry_path)
registry["views"] = {}
registry["web_maps"] = {}

for label, v_item, wm_item, _ in _all_items:
    key = label.lower()
    registry["views"][key] = {"item_id": v_item.id, "title": v_item.title}
    if wm_item:
        registry["web_maps"][key] = {"item_id": wm_item.id, "title": wm_item.title}

views.save_registry(_registry_path, registry)
print(f"Saved item registry to {Path(_registry_path).resolve()}")
print(json.dumps(registry, indent=2))

Saved item registry to /Users/rami8629/Library/CloudStorage/OneDrive-Esri/Demos & Blogs/ArcGIS Resources/ArcGIS as Geospatial AI Platofrm/Demos/2026/arcgis-ai-analysis/item_registry_staging.json
{
  "base_layer": {
    "item_id": "d7219e80879b42c98ddc7cda1a0369ff",
    "title": "Plant Identification AI STAGING",
    "url": "https://services.arcgis.com/LG9Yn2oFqZi5PnO5/arcgis/rest/services/Plant_Identification_AI_STAGING_202604221835/FeatureServer/0",
    "schema": "Plant Identification",
    "folder": null,
    "folder_name": "Plant Identification - Staging"
  },
  "views": {
    "collector": {
      "item_id": "5b3fece5247b4862ae7009f1aabd44b6",
      "title": "Plant Identification AI STAGING - Collector - STAGING"
    },
    "approver": {
      "item_id": "ed533487ea0c4877a71d14e591a40322",
      "title": "Plant Identification AI STAGING - Approver - STAGING"
    },
    "public": {
      "item_id": "367a69b3a3a747e5a56d4d9504b1d024",
      "title": "Plant Identification AI STAGING - 